## Análise de chamados - Bronze Layer

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"

df_chamados = spark.table(f"{bronze_schema}.chamados")
total = df_chamados.count()
display(df_chamados)

id_chamado,id_cliente,motivo,canal,resolvido,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera,tempo_atendimento,id_atendente,ingestion_datetime
76696,74719850215,Renegociação de dívida,atendimento inicial,Sim,null,null,null,49,441,4,2026-10-22T17:30:37.937Z
44377,92983951302,CONSULTA DE CONTRATO,Atendimento_Inicial,Sim,null,null,null,374,355,8,2026-10-22T17:30:37.937Z
34306,77740207578,Consulta de limite,ATENDIMENTO INICIAL,Sim,null,null,null,110,985,6,2026-10-22T17:30:37.937Z
52060,36276971799,Renegociação de dívida,Ura,Não,null,igual a hora de abertura,null,NULL,216,9,2026-10-22T17:30:37.937Z
25271,53061261807,Renegociação de dívida,Atendimento_Inicial,N,null,null,null,96,384,2,2026-10-22T17:30:37.937Z
88817,28669204533,Contratação de cartão adicional,Atendimento Inicial,Não,null,null,null,344,891,23,2026-10-22T17:30:37.937Z
89333,52402176464,RENEGOCIAÇÃO DE DÍVIDA,CHATBOT,N,null,igual a hora de abertura,null,NULL,104,3,2026-10-22T17:30:37.937Z
128376,99286903466,Compra não autorizada,Chatbot,Não,null,igual a hora de abertura,null,NULL,39,25,2026-10-22T17:30:37.937Z
14867,14853867391,Dúvidas gerais sobre programa de pontos,Chatbot,Não,null,igual a hora de abertura,null,NULL,118,17,2026-10-22T17:30:37.937Z
55810,23896919281,Contestação de fatura,CHATBOT,Não,null,igual a hora de abertura,null,NULL,110,5,2026-10-22T17:30:37.937Z


### Funções auxiliares

In [0]:
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]      # textos que na prática significam "sem valor"

def texto(c):
    """Coluna como string, sem espaços nas pontas."""
    return F.trim(F.col(c).cast("string"))

def eh_faltante(c):
    """Null de verdade, string vazia/só espaços, ou texto tipo 'NULL'."""
    return F.col(c).isNull() | (texto(c) == "") | F.upper(texto(c)).isin(TOKENS_NULOS)

def mostrar(df_, descricao):
    """Imprime a descrição e faz o display. Se o DataFrame estiver vazio, informa que não há casos."""
    if df_.isEmpty():
        print(f"\n{descricao}: nenhum caso encontrado")
    else:
        print(f"\n{descricao}")
        display(df_)

### 1. Duplicidade

In [0]:
colunas_negocio = [c for c in df_chamados.columns if c != "ingestion_datetime"]
distintas = df_chamados.select(*colunas_negocio).distinct().count()
print(f"Linhas excedentes por cópia exata: {total - distintas}")

ids_repetidos = (
    df_chamados.filter(~eh_faltante("id_chamado"))
      .groupBy(texto("id_chamado").alias("id_chamado"))
      .agg(
          F.count("*").alias("ocorrencias"),
          F.size(F.collect_set(F.struct(*[c for c in colunas_negocio if c != "id_chamado"]))).alias("versoes_distintas"),
      )
      .filter(F.col("ocorrencias") > 1)
)
n_ids = ids_repetidos.count()
linhas_envolvidas = ids_repetidos.agg(F.sum("ocorrencias")).first()[0] or 0
n_divergentes = ids_repetidos.filter(F.col("versoes_distintas") > 1).count()
print(f"id_chamado repetidos: {n_ids} ids em {linhas_envolvidas} linhas ({linhas_envolvidas - n_ids} linhas excedentes)")
print(f"  -> {n_divergentes} desses ids aparecem com conteúdo diferente entre as linhas")
mostrar(ids_repetidos.orderBy(F.desc("ocorrencias"), "id_chamado"),
        "ids de chamado repetidos (ocorrencias = nº de linhas; versoes_distintas = nº de conteúdos diferentes)")

Linhas excedentes por cópia exata: 2636
id_chamado repetidos: 2636 ids em 5272 linhas (2636 linhas excedentes)
  -> 0 desses ids aparecem com conteúdo diferente entre as linhas

ids de chamado repetidos (ocorrencias = nº de linhas; versoes_distintas = nº de conteúdos diferentes)


id_chamado,ocorrencias,versoes_distintas
100068,2,1
100109,2,1
100147,2,1
100186,2,1
100203,2,1
100241,2,1
100434,2,1
100555,2,1
100595,2,1
100609,2,1


In [0]:
# linhas completas dos ids repetidos (join sobre o id com o mesmo trim usado no agrupamento)
linhas_repetidas = (
    df_chamados
      .withColumn("id_chave", texto("id_chamado"))
      .join(ids_repetidos.select(F.col("id_chamado").alias("id_chave")), on="id_chave", how="inner")
      .drop("id_chave")
      .orderBy("id_chamado")
)
mostrar(linhas_repetidas, "Linhas completas dos id_chamado repetidos")


Linhas completas dos id_chamado repetidos


id_chamado,id_cliente,motivo,canal,resolvido,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera,tempo_atendimento,id_atendente,ingestion_datetime
100068,16861863398,Consulta de fatura,CHATBOT,Não,null,igual a hora de abertura,null,NULL,57,16,2026-10-22T17:30:37.937Z
100068,16861863398,Consulta de fatura,CHATBOT,Não,null,igual a hora de abertura,null,NULL,57,16,2026-10-22T17:30:37.937Z
100109,21741125306,Compra não autorizada,atendimento inicial,Não,null,null,null,770,726,7,2026-10-22T17:30:37.937Z
100109,21741125306,Compra não autorizada,atendimento inicial,Não,null,null,null,770,726,7,2026-10-22T17:30:37.937Z
100147,48407394243,Bloqueio de cartão,atendimento inicial,Sim,null,null,null,232,NULL,21,2026-10-22T17:30:37.937Z
100147,48407394243,Bloqueio de cartão,atendimento inicial,Sim,null,null,null,232,NULL,21,2026-10-22T17:30:37.937Z
100186,81217142116,Contestação de fatura,chat bot,Não,null,igual a hora de abertura,null,NULL,42,14,2026-10-22T17:30:37.937Z
100186,81217142116,Contestação de fatura,chat bot,Não,null,igual a hora de abertura,null,NULL,42,14,2026-10-22T17:30:37.937Z
100203,25548506878,Bloqueio de cartão,chatbot,Não,null,igual a hora de abertura,null,NULL,207,3,2026-10-22T17:30:37.937Z
100203,25548506878,Bloqueio de cartão,chatbot,Não,null,igual a hora de abertura,null,NULL,207,3,2026-10-22T17:30:37.937Z


### 2. Entradas faltantes por coluna

In [0]:
exprs = []
for c in df_chamados.columns:
    exprs += [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__nulo"),
        F.sum(F.when(texto(c) == "", 1).otherwise(0)).alias(f"{c}__vazio"),
        F.sum(F.when(F.upper(texto(c)).isin(TOKENS_NULOS), 1).otherwise(0)).alias(f"{c}__texto_nulo"),
    ]
r = df_chamados.agg(*exprs).first()

df_chamados_faltantes = (
    spark.createDataFrame(
        [(c, r[f"{c}__nulo"], r[f"{c}__vazio"], r[f"{c}__texto_nulo"]) for c in df_chamados.columns],
        "coluna string, nulo long, vazio long, texto_nulo long",
    )
    .withColumn("total_faltante", F.col("nulo") + F.col("vazio") + F.col("texto_nulo"))
    .withColumn("pct_faltante", F.round(F.col("total_faltante") * 100 / total, 2))
    .orderBy(F.desc("pct_faltante"))
)
mostrar(df_chamados_faltantes,
        "Faltantes por coluna: null real, string vazia e texto tipo 'NULL', com % sobre o total de linhas")


Faltantes por coluna: null real, string vazia e texto tipo 'NULL', com % sobre o total de linhas


coluna,nulo,vazio,texto_nulo,total_faltante,pct_faltante
hora_abertura_chamado,134474,0,0,134474,100.0
hora_finalizacao_atendimento,134474,0,0,134474,100.0
tempo_espera,0,0,77149,77149,57.37
hora_inicio_atendimento,57877,0,0,57877,43.04
tempo_atendimento,0,0,4541,4541,3.38
resolvido,2063,0,0,2063,1.53
id_chamado,0,0,0,0,0.0
id_cliente,0,0,0,0,0.0
motivo,0,0,0,0,0.0
canal,0,0,0,0,0.0


### 3. id_cliente x padrão de CPF (11 dígitos)

In [0]:
tipo = dict(df_chamados.dtypes)["id_cliente"]
if tipo != "string":
    print(f"ATENÇÃO: id_cliente está como {tipo}. Se for numérico, CPFs que começam com 0 "
          "perderam o zero à esquerda e vão cair em 'menos de 11 dígitos'.")

cpf = texto("id_cliente")
df_chamados_cpf = df_chamados.withColumn(
    "categoria_cpf",
    F.when(eh_faltante("id_cliente"), "faltante")
     .when(cpf.rlike(r"^\d{11}$"), "ok: 11 dígitos")
     .when(cpf.rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"), "formatado com . e -")
     .when(cpf.rlike(r"^\d{1,10}$"), "menos de 11 dígitos")
     .when(cpf.rlike(r"^\d{12,}$"), "mais de 11 dígitos")
     .otherwise("caracteres inválidos"),
)
mostrar(
    df_chamados_cpf.groupBy("categoria_cpf").count()
          .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
          .orderBy(F.desc("count")),
    "Distribuição de id_cliente por categoria de padrão de CPF",
)
mostrar(
    df_chamados_cpf.filter(~F.col("categoria_cpf").isin("ok: 11 dígitos", "faltante"))
          .groupBy("categoria_cpf", "id_cliente").count()
          .orderBy("categoria_cpf", F.desc("count")),
    "Valores de id_cliente fora do padrão, por categoria",
)


Distribuição de id_cliente por categoria de padrão de CPF


categoria_cpf,count,pct
ok: 11 dígitos,134474,100.0



Valores de id_cliente fora do padrão, por categoria: nenhum caso encontrado


### 4. Valores distintos: motivo, canal, resolvido

In [0]:
for c in ["motivo", "canal", "resolvido"]:
    contagem = df_chamados.groupBy(c).count().orderBy(F.desc("count"))
    mostrar(contagem, f"'{c}': valores distintos (grafia original) e nº de linhas de cada")


'motivo': valores distintos (grafia original) e nº de linhas de cada


motivo,count
Consulta de fatura,16337
Problema com aplicativo,14024
Consulta de limite,13589
Bloqueio de cartão,12471
Compra não autorizada,12311
Renegociação de dívida,11724
Contestação de fatura,10599
Desbloqueio de cartão,8380
Cancelamento de conta,5767
Consulta de contrato,5417



'canal': valores distintos (grafia original) e nº de linhas de cada


canal,count
URA,22920
Chatbot,15479
Atendimento Inicial,13432
Atendimento_Inicial,6854
atendimento inicial,6848
ATENDIMENTO INICIAL,6686
Atend. Inicial,6666
ura,5857
U.R.A,5842
Ura,5750



'resolvido': valores distintos (grafia original) e nº de linhas de cada


resolvido,count
Sim,63927
Não,60955
S,2128
null,2063
N,1939
1,1347
0,1290
Nao resolvido,825


### 5. Tempos (segundos): inteiro não negativo e 10 maiores

In [0]:
for c in ["tempo_espera", "tempo_atendimento"]:
    t = texto(c)
    df_chamados_t = df_chamados.withColumn(
        "categoria",
        F.when(eh_faltante(c), "faltante")
         .when(t.rlike(r"^\d+$"), "ok: inteiro >= 0")
         .when(t.rlike(r"^-\d+([.,]\d+)?$"), "negativo")
         .when(t.rlike(r"^\d+[.,]\d+$"), "decimal")
         .otherwise("não numérico"),
    )
    print(f"\n--- {c} ---")
    mostrar(
        df_chamados_t.groupBy("categoria").count()
            .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
            .orderBy(F.desc("count")),
        f"Distribuição de {c} por categoria de formato",
    )
    mostrar(
        df_chamados_t.filter(~F.col("categoria").isin("ok: inteiro >= 0", "faltante"))
            .groupBy("categoria", c).count()
            .orderBy("categoria", F.desc("count")),
        f"Valores inconsistentes de {c} (negativos, decimais ou não numéricos)",
    )
    mostrar(
        df_chamados_t.filter(F.col("categoria") == "ok: inteiro >= 0")
            .withColumn("segundos", t.cast("long"))
            .select("id_chamado", "canal", "segundos",
                    (F.round(F.col("segundos") / 60, 1)).alias("minutos"),
                    (F.round(F.col("segundos") / 3600, 2)).alias("horas"))
            .orderBy(F.desc("segundos"))
            .limit(10),
        f"10 maiores valores válidos de {c}",
    )


--- tempo_espera ---

Distribuição de tempo_espera por categoria de formato


categoria,count,pct
faltante,77149,57.37
ok: inteiro >= 0,57325,42.63



Valores inconsistentes de tempo_espera (negativos, decimais ou não numéricos): nenhum caso encontrado

10 maiores valores válidos de tempo_espera


id_chamado,canal,segundos,minutos,horas
125564,Atend. Inicial,999,16.7,0.28
131180,Atend. Inicial,999,16.7,0.28
113740,ATENDIMENTO ESPECIALIZADO,999,16.7,0.28
115896,Atendimento Inicial,999,16.7,0.28
96255,atendimento inicial,999,16.7,0.28
123350,Atend. Inicial,999,16.7,0.28
48738,Atendimento_Especializado,999,16.7,0.28
111449,Atend. Especializado,999,16.7,0.28
100327,Atendimento Especializado,999,16.7,0.28
50062,atendimento especializado,999,16.7,0.28



--- tempo_atendimento ---

Distribuição de tempo_atendimento por categoria de formato


categoria,count,pct
ok: inteiro >= 0,129933,96.62
faltante,4541,3.38



Valores inconsistentes de tempo_atendimento (negativos, decimais ou não numéricos): nenhum caso encontrado

10 maiores valores válidos de tempo_atendimento


id_chamado,canal,segundos,minutos,horas
126788,Atendimento_Inicial,999,16.7,0.28
121397,Atendimento Inicial,999,16.7,0.28
107444,Atend. Especializado,999,16.7,0.28
112482,atendimento inicial,999,16.7,0.28
129191,Atend. Especializado,999,16.7,0.28
98598,Atendimento Inicial,999,16.7,0.28
42369,Atendimento Inicial,999,16.7,0.28
79129,atendimento inicial,999,16.7,0.28
95256,Atendimento Inicial,999,16.7,0.28
79819,Atend. Inicial,999,16.7,0.28


### 6. id_atendente faltante x canal

In [0]:
n_sem_atendente = df_chamados.filter(eh_faltante("id_atendente")).count()
print(f"Linhas com id_atendente faltante: {n_sem_atendente} ({round(n_sem_atendente * 100 / total, 2)}%)")

sem_atendente_por_canal = (
    df_chamados.groupBy("canal")
      .agg(
          F.sum(F.when(eh_faltante("id_atendente"), 1).otherwise(0)).alias("sem_atendente"),
          F.count("*").alias("total_canal"),
      )
      .filter(F.col("sem_atendente") > 0)
      .withColumn("pct_do_canal", F.round(F.col("sem_atendente") * 100 / F.col("total_canal"), 2))
      .orderBy(F.desc("sem_atendente"))
)
mostrar(sem_atendente_por_canal,
        "Canais das linhas sem id_atendente (sem_atendente = linhas sem atendente no canal; "
        "pct_do_canal = % das linhas daquele canal que estão sem atendente)")

Linhas com id_atendente faltante: 0 (0.0%)

Canais das linhas sem id_atendente (sem_atendente = linhas sem atendente no canal; pct_do_canal = % das linhas daquele canal que estão sem atendente): nenhum caso encontrado


### 7. Linhas com canal faltante

In [0]:
mostrar(df_chamados.filter(eh_faltante("canal")),
        "Linhas em que canal é null, vazio ou texto tipo 'NULL'")


Linhas em que canal é null, vazio ou texto tipo 'NULL': nenhum caso encontrado


### 8. Diagnóstico: datas nas colunas categóricas

In [0]:
print("=== DIAGNÓSTICO: datas em motivo / canal / resolvido (bronze, antes da limpeza) ===")
PADRAO_DATA = r"\d{1,2}/\d{1,2}/\d{2,4}"
tem_data = {c: F.col(c).cast("string").rlike(PADRAO_DATA) for c in ["motivo", "canal", "resolvido"]}
for c, cond in tem_data.items():
    print(f"  {c}: {df_chamados.filter(cond).count()} linhas com data")
mostrar(
    df_chamados.filter(tem_data["motivo"] | tem_data["canal"] | tem_data["resolvido"]),
    "Linhas da bronze com data em motivo, canal ou resolvido.",
)

=== DIAGNÓSTICO: datas em motivo / canal / resolvido (bronze, antes da limpeza) ===
  motivo: 0 linhas com data
  canal: 0 linhas com data
  resolvido: 0 linhas com data

Linhas da bronze com data em motivo, canal ou resolvido.: nenhum caso encontrado


### Conclusões e tratamentos previstos para a Silver

| Problema encontrado | Evidência | Tratamento |
|---|---|---|
| Cópias exatas | 2.636 linhas idênticas (134.474 linhas para 131.838 chamados), sem versões divergentes | Silver: uma linha por `id_chamado` |
| Colunas de horário sem uso | `hora_abertura_chamado` e `hora_finalizacao_atendimento` 100% nulas. `hora_inicio_atendimento` nula em 43% e, quando preenchida, traz o texto "igual a hora de abertura" | Silver: colunas removidas. Os horários vêm da tabela `chamados_hora` |
| `tempo_espera` faltante | 77.149 textos `NULL` (57,37%). Os exemplos coincidem com "igual a hora de abertura" em URA e Chatbot | Silver: converter para null. Gold: usar o tempo informado, senão o calculado pelos horários, senão 0 |
| `tempo_atendimento` faltante | 4.541 textos `NULL` (3,38%) | Silver: converter para null |
| Tempos com teto em 999 | Os 10 maiores valores de `tempo_espera` e de `tempo_atendimento` são exatamente 999 segundos | A verificar: possível limite do sistema de origem |
| `canal` com grafias variadas | 19 grafias (URA / ura / U.R.A, Chatbot / chat bot, Atendimento_Inicial / Atend. Inicial etc.), nenhum nulo | Silver: mapear para URA, Chatbot, Atendimento Inicial e Atendimento Especializado |
| `motivo` com variação de caixa e acento | 37 grafias para os 14 motivos do case ("Consulta de fatura", "CONSULTA DE FATURA", "Bloqueio de cartao") | Silver: mapear para o nome padrão de cada motivo |
| `resolvido` sem padrão | Sim/Não, S/N, 1/0, "Nao resolvido" e 2.063 nulos (1,53%) | Silver: converter para booleano. Gold: nulo tratado como não resolvido |
| `id_atendente` | Nenhum faltante. Nos exemplos, chamados de URA e Chatbot também têm atendente | A verificar: atendente em canal de autosserviço |
| `id_cliente` | 100% com 11 dígitos | Nenhum |